In [ ]:
pip install xgboost shap optuna sentence-transformers scikit-learn pandas numpy matplotlib seaborn scipy

In [ ]:
pip install Groq

In [ ]:
# all imports in one place so we dont have to hunt for them later
# if something breaks its probably because cell 1 didnt finish properly

import pandas as pd
import numpy as np
import json
import hashlib
import time
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, f1_score

from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
import shap
import optuna

from groq import Groq

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_SEED = 42


GROQ_API_KEY = ""


client = Groq(api_key=GROQ_API_KEY)

print("imports done")
print("groq client ready")

In [ ]:

print("loading adult dataset.")

adult = fetch_openml(name="adult", version=2, as_frame=True)
df_adult = adult.frame

df_adult["class"] = df_adult["class"].str.strip()

print(f"adult loaded - shape: {df_adult.shape}")
print(f"class balance:\n{df_adult['class'].value_counts()}")


print("\nupload  hmeq.csv file when the picker appears")

from google.colab import files
uploaded = files.upload()

df_hmeq = pd.read_csv("hmeq.csv")


df_hmeq = df_hmeq.dropna()

print(f"hmeq loaded  shape after dropna: {df_hmeq.shape}")
print(f"class balance:\n{df_hmeq['BAD'].value_counts()}")



print("\n adult sample")
print(df_adult.head(3))

print("\n hmeq sample ")
print(df_hmeq.head(3))

In [ ]:
# take 1000 rows from each dataset
# stratified means keep the same class balance as the full dataset
# samples are saved to drive so always use the exact same rows

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
DRIVE_FOLDER = "/content/drive/MyDrive/llm_tabular_project"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

SAMPLE_PATH_ADULT = f"{DRIVE_FOLDER}/df_adult_sample.csv"
SAMPLE_PATH_HMEQ  = f"{DRIVE_FOLDER}/df_hmeq_sample.csv"


def stratified_subsample(df, target_col, n=1000, seed=RANDOM_SEED):
    _, sample = train_test_split(
        df,
        test_size=n / len(df),
        stratify=df[target_col],
        random_state=seed
    )
    # reset index so rows go 0,1,2
    sample = sample.reset_index(drop=True)
    balance = sample[target_col].value_counts(normalize=True).to_dict()
    print(f"sampled {len(sample)} rows - class balance: {balance}")
    return sample



if os.path.exists(SAMPLE_PATH_ADULT):
    df_adult_sample = pd.read_csv(SAMPLE_PATH_ADULT)
    print(f"adult sample loaded from drive - {len(df_adult_sample)} rows")
else:
    print("adult sample not found on drive - sampling fresh...")
    df_adult_sample = stratified_subsample(df_adult, target_col="class")
    df_adult_sample.to_csv(SAMPLE_PATH_ADULT, index=False)
    print(f"adult sample saved to drive")



if os.path.exists(SAMPLE_PATH_HMEQ):
    df_hmeq_sample = pd.read_csv(SAMPLE_PATH_HMEQ)
    print(f"hmeq sample loaded from drive - {len(df_hmeq_sample)} rows")
else:
    print("hmeq sample not found on drive - sampling fresh...")
    df_hmeq_sample = stratified_subsample(df_hmeq, target_col="BAD")
    df_hmeq_sample.to_csv(SAMPLE_PATH_HMEQ, index=False)
    print(f"hmeq sample saved to drive")



y_adult = (df_adult_sample["class"] == ">50K").astype(int).values
y_hmeq  = df_hmeq_sample["BAD"].astype(int).values

print(f"\nadult target  positive rate: {y_adult.mean():.2%}")
print(f"hmeq target   positive rate: {y_hmeq.mean():.2%}")
print(f"\nsamples saved to drive")

In [ ]:
# xgboost cant handle raw string columns like "occupation" or "education"
# so encode them as integers using ordinal encoder
# ordinal is fine for xgboost,it handles integer splits correctly
#do this BEFORE sending rows to the llm so the llm still sees readable labels
# but X_raw going into the model uses encoded integers


ADULT_CAT_COLS = [
    "workclass", "education", "marital-status",
    "occupation", "relationship", "race", "sex", "native-country"
]
ADULT_NUM_COLS = [
    "age", "fnlwgt", "education-num",
    "capital-gain", "capital-loss", "hours-per-week"
]

# hmeq only has 2 string columns
HMEQ_CAT_COLS = ["REASON", "JOB"]
HMEQ_NUM_COLS = [
    "LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG",
    "DELINQ", "CLAGE", "NINQ", "CLNO", "DEBTINC"
]




def preprocess_adult(df):
    # handle_unknown fills anything unseen with -1 instead of crashing
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    cat_encoded = enc.fit_transform(df[ADULT_CAT_COLS])
    num_vals    = df[ADULT_NUM_COLS].values.astype(float)
    # stack numeric and encoded categorical side by side
    # final shape is (n_rows, 14) - 6 numeric + 8 categorical
    return np.hstack([num_vals, cat_encoded])

def preprocess_hmeq(df):
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    cat_encoded = enc.fit_transform(df[HMEQ_CAT_COLS])
    num_vals    = df[HMEQ_NUM_COLS].values.astype(float)
    # final shape is (n_rows, 12) - 10 numeric + 2 categorical
    return np.hstack([num_vals, cat_encoded])




X_raw_adult = preprocess_adult(df_adult_sample)
X_raw_hmeq  = preprocess_hmeq(df_hmeq_sample)

print(f"X_raw_adult shape {X_raw_adult.shape}")
print(f"X_raw_hmeq shape  {X_raw_hmeq.shape}")


print(f"\nnans in adult raw {np.isnan(X_raw_adult).sum()}")
print(f"nans in hmeq raw-  {np.isnan(X_raw_hmeq).sum()}")

print("\npreprocessing done")

In [ ]:

PROMPTS_ADULT = {
    "career_trajectory": """You are a labour economist. Describe the likely career
    trajectory and financial behaviour patterns this profile suggests.
    Do not mention income or earnings directly.

    Output format:
    CAREER_STAGE: [early/mid/late/senior]
    STABILITY: [1-5]
    BEHAVIOUR_NOTE: [one sentence]""",

    "financial_risk": """You are a credit analyst. Describe the financial risk
    profile and spending behaviour patterns this individual likely exhibits,
    based on their demographic and employment characteristics.
    Do not mention income or creditworthiness directly.

    Output format:
    CAREER_STAGE: [early/mid/late/senior]
    STABILITY: [1-5]
    BEHAVIOUR_NOTE: [one sentence]""",

    "social_capital": """You are a sociologist. Describe the social capital,
    professional network strength, and economic mobility patterns typically
    associated with this demographic and occupational profile.
    Do not mention income directly.

    Output format:
    CAREER_STAGE: [early/mid/late/senior]
    STABILITY: [1-5]
    BEHAVIOUR_NOTE: [one sentence]"""
}

# hmeq prompts

PROMPTS_HMEQ = {
    "career_trajectory": """You are a labour economist. Based on this loan
    applicant's occupation category and employment history, describe their
    likely career stage, job stability, and professional trajectory.
    Do not mention loan repayment, default risk, or creditworthiness directly.

    Output format:
    CAREER_STAGE: [early/mid/late/senior]
    STABILITY: [1-5]
    BEHAVIOUR_NOTE: [one sentence]""",

    "financial_risk": """You are a behavioural economist. Based on this person's
    occupation, loan purpose, and credit history signals, describe their likely
    financial management patterns and attitude toward debt.
    Do not mention loan default or creditworthiness directly.

    Output format:
    CAREER_STAGE: [early/mid/late/senior]
    STABILITY: [1-5]
    BEHAVIOUR_NOTE: [one sentence]""",

    "social_capital": """You are a sociologist. Based on this person's occupation
    category and loan purpose, describe the social capital, financial literacy
    patterns, and economic stability typically associated with this profile.
    Do not mention loan repayment or default directly.

    Output format:
    CAREER_STAGE: [early/mid/late/senior]
    STABILITY: [1-5]
    BEHAVIOUR_NOTE: [one sentence]"""
}



CACHE_PATH_ADULT = "/content/drive/MyDrive/llm_tabular_project/llm_cache_adult.json"
CACHE_PATH_HMEQ  = "/content/drive/MyDrive/llm_tabular_project/llm_cache_hmeq.json"

def get_cache_key(prompt):
    return hashlib.md5(prompt.encode()).hexdigest()

def load_cache(path):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        # first run no cache file yet so start fresh
        return {}

def save_cache(cache, path):
    with open(path, "w") as f:
        json.dump(cache, f, indent=2)

def call_llm_cached(prompt, cache, cache_path):
    # check cache firstif  already called this exact prompt dont call again
    key = get_cache_key(prompt)
    if key in cache:
        return cache[key]

    # not in cache so actually call the api
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,   # low temp so responses are consistent
        max_tokens=150     # we only need a few lines back
    )
    result = response.choices[0].message.content

    # save to cache immediately so we dont lose it if something crashes
    cache[key] = result
    save_cache(cache, cache_path)

    time.sleep(0.5)
    return result


# load existing caches  empty dicts if first run
cache_adult = load_cache(CACHE_PATH_ADULT)
cache_hmeq  = load_cache(CACHE_PATH_HMEQ)

print(f"adult cache loaded - {len(cache_adult)} existing entries")
print(f"hmeq cache loaded  - {len(cache_hmeq)} existing entries")
print("\nprompts and cache ready")

In [ ]:
# this cell does api calls

from google.colab import drive


drive.mount("/content/drive", force_remount=False)


DRIVE_FOLDER = "/content/drive/MyDrive/llm_tabular_project"

import os
os.makedirs(DRIVE_FOLDER, exist_ok=True)


CACHE_PATH_ADULT = f"{DRIVE_FOLDER}/llm_cache_adult.json"
CACHE_PATH_HMEQ  = f"{DRIVE_FOLDER}/llm_cache_hmeq.json"

print(f"cache will be saved to {DRIVE_FOLDER}")


cache_adult = load_cache(CACHE_PATH_ADULT)
cache_hmeq  = load_cache(CACHE_PATH_HMEQ)



def format_row_adult(row):
    return (
        f"Age: {row['age']}, "
        f"Workclass: {row['workclass']}, "
        f"Education: {row['education']} ({row['education-num']} years), "
        f"Marital status: {row['marital-status']}, "
        f"Occupation: {row['occupation']}, "
        f"Relationship: {row['relationship']}, "
        f"Race: {row['race']}, "
        f"Sex: {row['sex']}, "
        f"Capital gain: {row['capital-gain']}, "
        f"Capital loss: {row['capital-loss']}, "
        f"Hours per week: {row['hours-per-week']}, "
        f"Country: {row['native-country']}"
    )

def format_row_hmeq(row):
    return (
        f"Loan amount: {row['LOAN']}, "
        f"Mortgage due: {row['MORTDUE']}, "
        f"Property value: {row['VALUE']}, "
        f"Loan reason: {row['REASON']}, "
        f"Job category: {row['JOB']}, "
        f"Years at job: {row['YOJ']}, "
        f"Derogatory reports: {row['DEROG']}, "
        f"Delinquencies: {row['DELINQ']}, "
        f"Oldest credit line age: {row['CLAGE']}, "
        f"Credit inquiries: {row['NINQ']}, "
        f"Credit lines: {row['CLNO']}, "
        f"Debt to income ratio: {row['DEBTINC']}"
    )




def run_llm_calls(df, format_row_fn, prompts_dict, cache, cache_path, label):
    results = {prompt_name: [] for prompt_name in prompts_dict}

    n_rows      = len(df)
    n_prompts   = len(prompts_dict)
    total_calls = n_rows * n_prompts
    calls_made  = 0
    cache_hits  = 0

    print(f"starting llm calls for {label}")
    print(f"rows {n_rows} | prompts: {n_prompts} | total calls {total_calls}")
    print(f"existing cache entries: {len(cache)} - these will be skipped")

    for i, row in df.iterrows():
        # print every 10 rows
        if i % 10 == 0:
            print(f"  row {i}/{n_rows} | cache hits so far {cache_hits}")

        row_str = format_row_fn(row)

        for prompt_name, prompt_text in prompts_dict.items():
            full_prompt = f"{row_str}\n\n{prompt_text}"

            key = get_cache_key(full_prompt)
            if key in cache:
                cache_hits += 1

            response = call_llm_cached(full_prompt, cache, cache_path)
            results[prompt_name].append(response)
            calls_made += 1

    print(f"\ndone - total calls attempted: {calls_made}")
    print(f"cache hits: {cache_hits} | actual api calls: {calls_made - cache_hits}")
    return results



print("\ncache status before starting ")
print(f"adult cache: {len(cache_adult)} entries")
print(f"hmeq cache:  {len(cache_hmeq)} entries")
print(f"adult calls still needed: {max(0, 3000 - len(cache_adult))}")
print(f"hmeq calls still needed:  {max(0, 3000 - len(cache_hmeq))}")
print(f"total api calls to make:  {max(0, 3000 - len(cache_adult)) + max(0, 3000 - len(cache_hmeq))}")



print("running adult llm calls")
responses_adult = run_llm_calls(
    df            = df_adult_sample,
    format_row_fn = format_row_adult,
    prompts_dict  = PROMPTS_ADULT,
    cache         = cache_adult,
    cache_path    = CACHE_PATH_ADULT,
    label         = "adult"
)


print("running hmeq llm calls.")
responses_hmeq = run_llm_calls(
    df            = df_hmeq_sample,
    format_row_fn = format_row_hmeq,
    prompts_dict  = PROMPTS_HMEQ,
    cache         = cache_hmeq,
    cache_path    = CACHE_PATH_HMEQ,
    label         = "hmeq"
)

print("\nall llm calls done")
print(f"adult cache now has {len(cache_adult)} entries")
print(f"hmeq cache now has:  {len(cache_hmeq)} entries")

In [ ]:
# the llm responses are raw text like:
# CAREER_STAGE: mid
# STABILITY: 3
# BEHAVIOUR_NOTE: This person likely manages finances conservatively

#  need to pull those values out and turn them into numbers
# also track how many responses failed to parse to can catch silent bias

def parse_llm_response(text):
    # start with defaults in case parsing fails
    result = {
        "career_stage"   : "unknown",
        "stability"      : 3,         # middle value as fallback
        "behaviour_note" : "",
        "parse_ok"       : True
    }
    found = {"stage": False, "stab": False, "note": False}

    for line in text.strip().split("\n"):
        line = line.strip()

        if line.startswith("CAREER_STAGE:"):
            val = line.split(":", 1)[1].strip().lower()
            if val in {"early", "mid", "late", "senior"}:
                result["career_stage"] = val
                found["stage"] = True

        elif line.startswith("STABILITY:"):
            try:
                # take first character only in case llm writes "3/5" or "3 out of 5"
                val = int(line.split(":", 1)[1].strip()[0])
                if 1 <= val <= 5:
                    result["stability"] = val
                    found["stab"] = True
            except (ValueError, IndexError):
                pass

        elif line.startswith("BEHAVIOUR_NOTE:"):
            result["behaviour_note"] = line.split(":", 1)[1].strip()
            found["note"] = True

    # if any field missing mark as failed parse
    if not all(found.values()):
        result["parse_ok"] = False

    return result


def report_parse_quality(parsed, label=""):
    # always run this before training, silent failures are a hidden bias risk
    n = len(parsed)
    failures      = [p for p in parsed if not p["parse_ok"]]
    unknown_stage = [p for p in parsed if p["career_stage"] == "unknown"]

    print(f"\n parse quality [{label}] ")
    print(f"total     {n}")
    print(f"failures  {len(failures)} ({len(failures)/n:.1%})")
    print(f"unknown stage  {len(unknown_stage)} ({len(unknown_stage)/n:.1%})")

    if len(failures) / n > 0.05:
        print("more than 5% parse failures ")
        # print a few examples to see whats going wrong
        for p in failures[:3]:
            print(f"  stage={p['career_stage']}, stab={p['stability']}, "
                  f"note='{p['behaviour_note'][:60]}'")
    else:
        print("parse failure rate acceptable under 5%")


# responses_adult and responses_hmeq are dicts with 3 keys (one per prompt)
# we use career_trajectory as the main prompt for features
# the other two are used in the prompt ablation section later

print("parsing adult responses")
parsed_adult = [parse_llm_response(r) for r in responses_adult["career_trajectory"]]
report_parse_quality(parsed_adult, label="adult career_trajectory")

print("\nparsing hmeq responses")
parsed_hmeq = [parse_llm_response(r) for r in responses_hmeq["career_trajectory"]]
report_parse_quality(parsed_hmeq, label="hmeq career_trajectory")




# one hot encode career stage - 4 possible values
# unknown maps to all zeros which is fine as a fallback
STAGE_VALS = ["early", "mid", "late", "senior"]

def make_stage_ohe(parsed):
    # shape (n_rows, 4)
    ohe = np.zeros((len(parsed), len(STAGE_VALS)))
    for i, p in enumerate(parsed):
        if p["career_stage"] in STAGE_VALS:
            ohe[i, STAGE_VALS.index(p["career_stage"])] = 1
    return ohe

def make_stability_col(parsed):
    # shape (n_rows, 1)
    return np.array([p["stability"] for p in parsed]).reshape(-1, 1)


stage_ohe_adult   = make_stage_ohe(parsed_adult)
stability_adult   = make_stability_col(parsed_adult)

stage_ohe_hmeq    = make_stage_ohe(parsed_hmeq)
stability_hmeq    = make_stability_col(parsed_hmeq)

print(f"\nstage_ohe_adult shape    {stage_ohe_adult.shape}")
print(f"stability_adult shape    {stability_adult.shape}")
print(f"stage_ohe_hmeq shape    {stage_ohe_hmeq.shape}")
print(f"stability_hmeq shape  {stability_hmeq.shape}")

# quick sanity check on stability values
print(f"\nadult stability  mean: {stability_adult.mean():.2f}  "
      f"min: {stability_adult.min()}  max: {stability_adult.max()}")
print(f"hmeq stability   mean: {stability_hmeq.mean():.2f}  "
      f"min: {stability_hmeq.min()}  max: {stability_hmeq.max()}")

print("\nparsing done")

In [ ]:


print("loading sentence transformer model")
encoder = SentenceTransformer("all-MiniLM-L6-v2")
print("model loaded")


def get_raw_embeddings(parsed, label=""):
    notes = [
        p["behaviour_note"] if p["behaviour_note"] else "no information available"
        for p in parsed
    ]

    print(f"\nencoding {len(notes)} notes for {label}")
    embeddings = encoder.encode(notes, batch_size=64, show_progress_bar=True)
    print(f"embeddings shape- {embeddings.shape}")
    return embeddings


# encode both datasets
# shape will be (1000, 384) for each
raw_emb_adult = get_raw_embeddings(parsed_adult, label="adult")
raw_emb_hmeq  = get_raw_embeddings(parsed_hmeq,  label="hmeq")

# check no nans in embeddings
print(f"\nnans in adult embeddings {np.isnan(raw_emb_adult).sum()}")
print(f"nans in hmeq embeddings:  {np.isnan(raw_emb_hmeq).sum()}")

# - values should be small floats
print(f"\nadult embeddings- mean: {raw_emb_adult.mean():.4f}  "
      f"std: {raw_emb_adult.std():.4f}")
print(f"hmeq embeddings-mean: {raw_emb_hmeq.mean():.4f}  "
      f"std: {raw_emb_hmeq.std():.4f}")

print("\nembeddings done")

In [ ]:

def plot_pca_sensitivity(embeddings, X_raw, y, label="dataset"):
    max_comp = min(100, embeddings.shape[0] - 1, embeddings.shape[1])

    # fit pca on full data just to see variance curve
    pca_check = PCA(n_components=max_comp).fit(embeddings)
    cumvar = np.cumsum(pca_check.explained_variance_ratio_)

    # find how many components we need for 90% and 95% variance
    n_90 = int(np.argmax(cumvar >= 0.90)) + 1
    n_95 = int(np.argmax(cumvar >= 0.95)) + 1

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    # left plot - variance retention curve
    ax1.plot(cumvar, linewidth=2, color="#2563eb")
    ax1.axhline(0.90, color="orange", linestyle="--",
                label=f"90% variance at {n_90} components")
    ax1.axhline(0.95, color="red", linestyle="--",
                label=f"95% variance at {n_95} components")
    ax1.set_xlabel("number of pca components")
    ax1.set_ylabel("cumulative explained variance")
    ax1.set_title(f"embedding variance retention - {label}")
    ax1.legend()

    print(f"[{label}] 5 components retains {cumvar[4]:.1%}")
    print(f"[{label}] 90% variance point     {n_90} components")
    print(f"[{label}] 95% variance point {n_95} components")

    # right plot - auc vs number of components
    # pca fit on full data here intentionally - this is just for picking n_components
    # not for reporting final results
    component_options = [5, 10, 15, 20, 30, 50]
    auc_scores = []
    pos_ratio = (y == 0).sum() / (y == 1).sum()

    for n in component_options:
        pca_exp = PCA(n_components=n, random_state=RANDOM_SEED)
        emb_r   = pca_exp.fit_transform(embeddings)
        X_t     = np.hstack([X_raw, emb_r])
        scores  = cross_val_score(
            XGBClassifier(scale_pos_weight=pos_ratio, random_state=RANDOM_SEED),
            X_t, y,
            cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED),
            scoring="roc_auc"
        )
        auc_scores.append(scores.mean())
        print(f"  n={n:3d} components  AUC {scores.mean():.4f}  "
              f"(retains {np.sum(pca_check.explained_variance_ratio_[:n]):.1%} variance)")

    best_n = component_options[int(np.argmax(auc_scores))]

    ax2.plot(component_options, auc_scores, "o-", linewidth=2, color="#16a34a")
    ax2.axvline(best_n, color="red", linestyle="--",
                label=f"best: {best_n} components")
    ax2.set_xlabel("number of pca components")
    ax2.set_ylabel("hybrid auc (exploratory 5-fold cv)")
    ax2.set_title(f"auc vs compression - {label} (exploratory only)")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f"{DRIVE_FOLDER}/pca_sensitivity_{label}.png", dpi=150)
    plt.show()
    print(f"best n_components for {label}: {best_n}\n")
    return best_n


# run for both datasets
print("running pca sensitivity for adult")
best_n_adult = plot_pca_sensitivity(
    embeddings = raw_emb_adult,
    X_raw      = X_raw_adult,
    y          = y_adult,
    label      = "adult"
)

print("running pca sensitivity for hmeq")
best_n_hmeq = plot_pca_sensitivity(
    embeddings = raw_emb_hmeq,
    X_raw      = X_raw_hmeq,
    y          = y_hmeq,
    label      = "hmeq"
)

print(f"chosen n_pca_components  adult: {best_n_adult}  hmeq: {best_n_hmeq}")


In [ ]:

def nested_cv_variant(
    X_raw,
    raw_embeddings,
    stage_ohe,
    stability_col,
    y,
    n_outer   = 5,
    n_repeats = 3,
    n_inner   = 5,
    n_pca     = 20,
    n_trials  = 30,
    label     = ""
):
    outer_cv  = RepeatedStratifiedKFold(
        n_splits   = n_outer,
        n_repeats  = n_repeats,
        random_state = RANDOM_SEED
    )

    outer_auc      = []
    outer_f1       = []
    all_best_params = []

    for fold_i, (train_idx, test_idx) in enumerate(outer_cv.split(X_raw, y)):
        print(f"[{label}] fold {fold_i + 1}/{n_outer * n_repeats}")

        y_tr, y_te = y[train_idx], y[test_idx]

        # skip pca if embeddings are placeholder zeros
        # baseline passes np.zeros((N,1)) so  guard against that here
        use_embeddings = raw_embeddings.shape[1] > 1

        if use_embeddings:
            pca_outer = PCA(n_components=n_pca, random_state=RANDOM_SEED)
            emb_tr    = pca_outer.fit_transform(raw_embeddings[train_idx])
            emb_te    = pca_outer.transform(raw_embeddings[test_idx])
        else:
            emb_tr = np.zeros((len(train_idx), 0))
            emb_te = np.zeros((len(test_idx),  0))

        # build structured feature matrices - raw + stage ohe + stability
        X_struct_tr = np.hstack([X_raw[train_idx], stage_ohe[train_idx],
                                  stability_col[train_idx]])
        X_struct_te = np.hstack([X_raw[test_idx],  stage_ohe[test_idx],
                                  stability_col[test_idx]])

        # hybrid adds the pca embeddings on top
        X_hybrid_tr = np.hstack([X_struct_tr, emb_tr])
        X_hybrid_te = np.hstack([X_struct_te, emb_te])

        pos_ratio_tr = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

        raw_emb_tr = raw_embeddings[train_idx]

        def objective(trial):
            params = {
                "n_estimators"     : trial.suggest_int("n_estimators", 100, 400),
                "max_depth"        : trial.suggest_int("max_depth", 3, 7),
                "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
                "scale_pos_weight" : trial.suggest_float("scale_pos_weight", 1.0, pos_ratio_tr),
            }

            inner_cv     = StratifiedKFold(n_splits=n_inner, shuffle=True,
                                            random_state=RANDOM_SEED)
            inner_scores = []

            for in_tr, in_val in inner_cv.split(X_struct_tr, y_tr):
                if use_embeddings:
                    # inner pca fit on inner train raw embeddings only
                    # NOT on the outer pca output ,that would be pca on pca
                    pca_inner  = PCA(n_components=n_pca, random_state=RANDOM_SEED)
                    emb_in_tr  = pca_inner.fit_transform(raw_emb_tr[in_tr])
                    emb_in_val = pca_inner.transform(raw_emb_tr[in_val])
                else:
                    emb_in_tr  = np.zeros((len(in_tr),  0))
                    emb_in_val = np.zeros((len(in_val), 0))

                Xi_tr  = np.hstack([X_struct_tr[in_tr],  emb_in_tr])
                Xi_val = np.hstack([X_struct_tr[in_val], emb_in_val])

                m = XGBClassifier(**params, random_state=RANDOM_SEED,
                                   eval_metric="auc")
                m.fit(Xi_tr, y_tr[in_tr])
                prob = m.predict_proba(Xi_val)[:, 1]
                inner_scores.append(roc_auc_score(y_tr[in_val], prob))

            return np.mean(inner_scores)

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_params = study.best_params
        all_best_params.append(best_params)

        best_model = XGBClassifier(**best_params, random_state=RANDOM_SEED,
                                    eval_metric="auc")
        best_model.fit(X_hybrid_tr, y_tr)

        prob  = best_model.predict_proba(X_hybrid_te)[:, 1]
        preds = (prob > 0.5).astype(int)
        auc   = roc_auc_score(y_te, prob)
        f1    = f1_score(y_te, preds, average="weighted")

        outer_auc.append(auc)
        outer_f1.append(f1)
        print(f"  AUC={auc:.4f}  F1={f1:.4f}")

    print(f"\n[{label}] AUC: {np.mean(outer_auc):.4f} +/- {np.std(outer_auc):.4f}")
    print(f"[{label}] F1 : {np.mean(outer_f1):.4f} +/- {np.std(outer_f1):.4f}")
    return outer_auc, all_best_params



N_adult = len(y_adult)
N_hmeq  = len(y_hmeq)


print("ADULT - BASELINE (raw features only)")

adult_baseline_scores, adult_baseline_params = nested_cv_variant(
    X_raw          = X_raw_adult,
    raw_embeddings = np.zeros((N_adult, 1)),
    stage_ohe      = np.zeros((N_adult, 4)),
    stability_col  = np.zeros((N_adult, 1)),
    y              = y_adult,
    n_pca          = best_n_adult,
    label          = "adult_baseline"
)


print("ADULT - LLM ONLY (embeddings only)")

adult_llm_scores, adult_llm_params = nested_cv_variant(
    X_raw          = np.zeros((N_adult, 1)),
    raw_embeddings = raw_emb_adult,
    stage_ohe      = stage_ohe_adult,
    stability_col  = stability_adult,
    y              = y_adult,
    n_pca          = best_n_adult,
    label          = "adult_llm_only"
)


print("ADULT - HYBRID (raw + llm)")

adult_hybrid_scores, adult_hybrid_params = nested_cv_variant(
    X_raw          = X_raw_adult,
    raw_embeddings = raw_emb_adult,
    stage_ohe      = stage_ohe_adult,
    stability_col  = stability_adult,
    y              = y_adult,
    n_pca          = best_n_adult,
    label          = "adult_hybrid"
)


print("HMEQ - BASELINE (raw features only)")

hmeq_baseline_scores, hmeq_baseline_params = nested_cv_variant(
    X_raw          = X_raw_hmeq,
    raw_embeddings = np.zeros((N_hmeq, 1)),
    stage_ohe      = np.zeros((N_hmeq, 4)),
    stability_col  = np.zeros((N_hmeq, 1)),
    y              = y_hmeq,
    n_pca          = best_n_hmeq,
    label          = "hmeq_baseline"
)


print("HMEQ - LLM ONLY (embeddings only)")

hmeq_llm_scores, hmeq_llm_params = nested_cv_variant(
    X_raw          = np.zeros((N_hmeq, 1)),
    raw_embeddings = raw_emb_hmeq,
    stage_ohe      = stage_ohe_hmeq,
    stability_col  = stability_hmeq,
    y              = y_hmeq,
    n_pca          = best_n_hmeq,
    label          = "hmeq_llm_only"
)


print("HMEQ - HYBRID (raw + llm)")

hmeq_hybrid_scores, hmeq_hybrid_params = nested_cv_variant(
    X_raw          = X_raw_hmeq,
    raw_embeddings = raw_emb_hmeq,
    stage_ohe      = stage_ohe_hmeq,
    stability_col  = stability_hmeq,
    y              = y_hmeq,
    n_pca          = best_n_hmeq,
    label          = "hmeq_hybrid"
)

print("\nall nested cv done")

In [ ]:

def significance_report(baseline_scores, hybrid_scores, label=""):
    a = np.array(baseline_scores)
    b = np.array(hybrid_scores)

    # wilcoxon needs paired scores same 15 folds for both variants
    stat, p = wilcoxon(a, b)

    # cohen's d - pooled std version
    pooled_std = np.sqrt((a.std()**2 + b.std()**2) / 2)
    d = (b.mean() - a.mean()) / pooled_std

    # standard thresholds for effect size magnitude
    magnitude = (
        "negligible" if abs(d) < 0.2 else
        "small"      if abs(d) < 0.5 else
        "medium"     if abs(d) < 0.8 else
        "large"
    )

    print(f"\nsignificance report [{label}] ")
    print(f"n paired scores  : {len(a)}  (3 repeats x 5 folds)")
    print(f"baseline AUC     : {a.mean():.4f} +/- {a.std():.4f}")
    print(f"hybrid AUC       : {b.mean():.4f} +/- {b.std():.4f}")
    print(f"delta            : {b.mean() - a.mean():+.4f}")
    print(f"wilcoxon p       : {p:.4f}  {'significant p<0.05' if p < 0.05 else 'not significant'}")
    print(f"cohens d         : {d:.3f}  ({magnitude} effect)")

    return p, d




print("ADULT SIGNIFICANCE TESTS")


p_adult_hybrid, d_adult_hybrid = significance_report(
    baseline_scores = adult_baseline_scores,
    hybrid_scores   = adult_hybrid_scores,
    label           = "adult hybrid vs baseline"
)

p_adult_llm, d_adult_llm = significance_report(
    baseline_scores = adult_baseline_scores,
    hybrid_scores   = adult_llm_scores,
    label           = "adult llm only vs baseline"
)


print("HMEQ SIGNIFICANCE TESTS")


p_hmeq_hybrid, d_hmeq_hybrid = significance_report(
    baseline_scores = hmeq_baseline_scores,
    hybrid_scores   = hmeq_hybrid_scores,
    label           = "hmeq hybrid vs baseline"
)

p_hmeq_llm, d_hmeq_llm = significance_report(
    baseline_scores = hmeq_baseline_scores,
    hybrid_scores   = hmeq_llm_scores,
    label           = "hmeq llm only vs baseline"
)


print("FULL RESULTS SUMMARY")

print(f"\n{'dataset':<10} {'variant':<12} {'AUC mean':<12} {'AUC std':<10} {'p value':<10} {'cohens d':<10} {'significant'}")

rows = [
    ("adult",  "baseline", adult_baseline_scores, "-",          "-"),
    ("adult",  "llm only", adult_llm_scores,      f"{p_adult_llm:.4f}",    f"{d_adult_llm:.3f}"),
    ("adult",  "hybrid",   adult_hybrid_scores,   f"{p_adult_hybrid:.4f}", f"{d_adult_hybrid:.3f}"),
    ("hmeq",   "baseline", hmeq_baseline_scores,  "-",          "-"),
    ("hmeq",   "llm only", hmeq_llm_scores,       f"{p_hmeq_llm:.4f}",    f"{d_hmeq_llm:.3f}"),
    ("hmeq",   "hybrid",   hmeq_hybrid_scores,    f"{p_hmeq_hybrid:.4f}", f"{d_hmeq_hybrid:.3f}"),
]

for dataset, variant, scores, p_val, d_val in rows:
    s = np.array(scores)
    sig = ""
    if p_val != "-":
        sig = "yes" if float(p_val) < 0.05 else "no"
    print(f"{dataset:<10} {variant:<12} {s.mean():<12.4f} {s.std():<10.4f} {p_val:<10} {d_val:<10} {sig}")

In [ ]:

def refit_for_interpretability(
    X_struct,
    raw_embeddings,
    y,
    all_best_params,
    n_pca,
    label=""
):

    median_params = {}
    for key in all_best_params[0]:
        vals = [p[key] for p in all_best_params]
        if isinstance(vals[0], int):
            median_params[key] = int(np.median(vals))
        else:
            median_params[key] = float(np.median(vals))

    print(f"[{label}] median params: {median_params}")


    idx = np.arange(len(y))
    tr_idx, te_idx = train_test_split(
        idx, test_size=0.2, stratify=y, random_state=RANDOM_SEED
    )

    use_embeddings = raw_embeddings.shape[1] > 1
    if use_embeddings:
        pca = PCA(n_components=n_pca, random_state=RANDOM_SEED)
        emb_tr = pca.fit_transform(raw_embeddings[tr_idx])
        emb_te = pca.transform(raw_embeddings[te_idx])
    else:
        emb_tr = np.zeros((len(tr_idx), 0))
        emb_te = np.zeros((len(te_idx), 0))

    X_tr = np.hstack([X_struct[tr_idx], emb_tr])
    X_te = np.hstack([X_struct[te_idx], emb_te])

    model = XGBClassifier(**median_params, random_state=RANDOM_SEED,
                           eval_metric="auc")
    model.fit(X_tr, y[tr_idx])

    holdout_auc = roc_auc_score(y[te_idx], model.predict_proba(X_te)[:, 1])
    print(f"[{label}] holdout AUC: {holdout_auc:.4f} (interpretability only - not the reported number)")

    return model, X_te, y[te_idx]




X_struct_adult = np.hstack([X_raw_adult, stage_ohe_adult, stability_adult])
X_struct_hmeq  = np.hstack([X_raw_hmeq,  stage_ohe_hmeq,  stability_hmeq])

print("refitting adult hybrid model for shap")
model_adult, X_te_adult, y_te_adult = refit_for_interpretability(
    X_struct       = X_struct_adult,
    raw_embeddings = raw_emb_adult,
    y              = y_adult,
    all_best_params = adult_hybrid_params,
    n_pca          = best_n_adult,
    label          = "adult_hybrid"
)

print("\nrefitting hmeq hybrid model for shap.")
model_hmeq, X_te_hmeq, y_te_hmeq = refit_for_interpretability(
    X_struct        = X_struct_hmeq,
    raw_embeddings  = raw_emb_hmeq,
    y               = y_hmeq,
    all_best_params = hmeq_hybrid_params,
    n_pca           = best_n_hmeq,
    label           = "hmeq_hybrid"
)

print("\nrefit done ...models ready for shap")

In [ ]:


def build_feature_names(X_raw_cols, n_pca, dataset="adult"):


    if dataset == "adult":
        raw_names = ADULT_NUM_COLS + ADULT_CAT_COLS
    else:
        raw_names = HMEQ_NUM_COLS + HMEQ_CAT_COLS

    # stage one hot columns
    stage_names = [f"llm_stage_{s}" for s in ["early", "mid", "late", "senior"]]

    # stability score
    stab_name = ["llm_stability"]

    # pca embedding columns
    pca_names = [f"llm_pca_{i}" for i in range(n_pca)]

    return raw_names + stage_names + stab_name + pca_names


def shap_analysis(model, X_test, feature_names, label="", drive_folder=None):
    print(f"running shap for {label}.")

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_values, X_test,
        feature_names=feature_names,
        plot_type="bar",
        show=False,
        max_display=20
    )
    plt.title(f"shap feature importance - {label}")
    plt.tight_layout()
    if drive_folder:
        plt.savefig(f"{drive_folder}/shap_bar_{label}.png", dpi=150)
    plt.show()


    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_values, X_test,
        feature_names=feature_names,
        show=False,
        max_display=20
    )
    plt.title(f"shap beeswarm - {label}")
    plt.tight_layout()
    if drive_folder:
        plt.savefig(f"{drive_folder}/shap_beeswarm_{label}.png", dpi=150)
    plt.show()

    all_imp   = np.abs(shap_values).mean(axis=0)
    rank_order = np.argsort(-all_imp)
    rank_lookup = {feat_idx: rank + 1 for rank, feat_idx in enumerate(rank_order)}

    llm_idx = [i for i, n in enumerate(feature_names) if n.startswith("llm_")]

    print(f"\n[{label}] llm feature ranks out of {len(feature_names)} total features:")
    for i in llm_idx:
        print(f"  {feature_names[i]:<35} rank={rank_lookup[i]:3d}  "
              f"mean shap={all_imp[i]:.4f}")

    return shap_values, all_imp

feature_names_adult = build_feature_names(
    X_raw_cols = None,
    n_pca      = best_n_adult,
    dataset    = "adult"
)

feature_names_hmeq = build_feature_names(
    X_raw_cols = None,
    n_pca      = best_n_hmeq,
    dataset    = "hmeq"
)


print(f"adult feature names {len(feature_names_adult)} | X_te columns: {X_te_adult.shape[1]}")
print(f"hmeq feature names  {len(feature_names_hmeq)} | X_te columns:  {X_te_hmeq.shape[1]}")


print("\n")
shap_values_adult, imp_adult = shap_analysis(
    model         = model_adult,
    X_test        = X_te_adult,
    feature_names = feature_names_adult,
    label         = "adult_hybrid",
    drive_folder  = DRIVE_FOLDER
)

print("\n")
shap_values_hmeq, imp_hmeq = shap_analysis(
    model         = model_hmeq,
    X_test        = X_te_hmeq,
    feature_names = feature_names_hmeq,
    label         = "hmeq_hybrid",
    drive_folder  = DRIVE_FOLDER
)

In [ ]:

def disagreement_analysis(
    X_raw_test,
    X_hybrid_test,
    y_test,
    model_base,
    model_hybrid,
    feature_names_hybrid,
    label=""
):
    base_probs   = model_base.predict_proba(X_raw_test)[:, 1]
    hybrid_probs = model_hybrid.predict_proba(X_hybrid_test)[:, 1]
    base_preds   = (base_probs > 0.5).astype(int)
    hybrid_preds = (hybrid_probs > 0.5).astype(int)

    disagree = base_preds != hybrid_preds
    n_dis    = disagree.sum()

    print(f"\ndisagreement analysis [{label}] ")
    print(f"total test samples  {len(y_test)}")
    print(f"disagreements      : {n_dis} ({disagree.mean():.1%})")

    if n_dis < 10:
        print("too few disagreements for meaningful analysis skipping")
        return

    y_d   = y_test[disagree]
    b_d   = base_preds[disagree]
    h_d   = hybrid_preds[disagree]

    base_acc   = (b_d == y_d).mean()
    hybrid_acc = (h_d == y_d).mean()

    print(f"\non disagreement samples (n={n_dis})")
    print(f"  baseline correct : {base_acc:.1%}")
    print(f"  hybrid correct   : {hybrid_acc:.1%}")

    # find stability column index in hybrid feature matrix
    # layout is raw | stage_ohe (4) | stability (1) | pca embeddings
    # so stability is right after the raw features and 4 ohe columns
    stab_idx = len([n for n in feature_names_hybrid
                    if not n.startswith("llm_")]) + 4

    wins   = np.where(disagree)[0][h_d == y_d]
    losses = np.where(disagree)[0][h_d != y_d]

    if len(wins) > 0 and len(losses) > 0:
        sw = X_hybrid_test[wins,   stab_idx].mean()
        sl = X_hybrid_test[losses, stab_idx].mean()
        print(f"\nllm stability on hybrid wins  (n={len(wins):3d}): {sw:.2f}")
        print(f"llm stability on hybrid losses (n={len(losses):3d}): {sl:.2f}")
        if abs(sw - sl) > 0.3:
            print("stability correlates with hybrid correct calls")
        else:
            print("no strong stability difference not clearly llm driven")

    # bar chart showing accuracy on disagreement samples
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(
        ["baseline\n(on disagreements)", "hybrid\n(on disagreements)"],
        [base_acc, hybrid_acc],
        color=["#94a3b8", "#2563eb"],
        width=0.5
    )
    ax.set_ylabel("accuracy")
    ax.set_title(f"who is right when models disagree - {label}  n={n_dis}")
    ax.set_ylim(0, 1)
    for i, v in enumerate([base_acc, hybrid_acc]):
        ax.text(i, v + 0.02, f"{v:.1%}", ha="center", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{DRIVE_FOLDER}/disagreement_{label}.png", dpi=150)
    plt.show()

# refit baseline models the same way as hybrid - 80/20 holdout median params

print("refitting baseline models for disagreement analysis...")

# adult baseline refit
model_adult_base, X_te_adult_base, _ = refit_for_interpretability(
    X_struct        = X_raw_adult,    # raw features only no llm columns
    raw_embeddings  = np.zeros((N_adult, 1)),
    y               = y_adult,
    all_best_params = adult_baseline_params,
    n_pca           = best_n_adult,
    label           = "adult_baseline"
)

# hmeq baseline refit
model_hmeq_base, X_te_hmeq_base, _ = refit_for_interpretability(
    X_struct        = X_raw_hmeq,
    raw_embeddings  = np.zeros((N_hmeq, 1)),
    y               = y_hmeq,
    all_best_params = hmeq_baseline_params,
    n_pca           = best_n_hmeq,
    label           = "hmeq_baseline"
)



print("\n")
disagreement_analysis(
    X_raw_test      = X_te_adult_base,
    X_hybrid_test   = X_te_adult,
    y_test          = y_te_adult,
    model_base      = model_adult_base,
    model_hybrid    = model_adult,
    feature_names_hybrid = feature_names_adult,
    label           = "adult"
)

print("\n")
disagreement_analysis(
    X_raw_test      = X_te_hmeq_base,
    X_hybrid_test   = X_te_hmeq,
    y_test          = y_te_hmeq,
    model_base      = model_hmeq_base,
    model_hybrid    = model_hmeq,
    feature_names_hybrid = feature_names_hmeq,
    label           = "hmeq"
)

In [ ]:

print("parsing remaining prompt framings")


parsed_adult_fin = [parse_llm_response(r) for r in responses_adult["financial_risk"]]
parsed_adult_soc = [parse_llm_response(r) for r in responses_adult["social_capital"]]

report_parse_quality(parsed_adult_fin, label="adult financial_risk")
report_parse_quality(parsed_adult_soc, label="adult social_capital")


parsed_hmeq_fin = [parse_llm_response(r) for r in responses_hmeq["financial_risk"]]
parsed_hmeq_soc = [parse_llm_response(r) for r in responses_hmeq["social_capital"]]

report_parse_quality(parsed_hmeq_fin, label="hmeq financial_risk")
report_parse_quality(parsed_hmeq_soc, label="hmeq social_capital")


print("\ngetting embeddings for remaining framings.")

raw_emb_adult_fin = get_raw_embeddings(parsed_adult_fin, label="adult financial_risk")
raw_emb_adult_soc = get_raw_embeddings(parsed_adult_soc, label="adult social_capital")

raw_emb_hmeq_fin  = get_raw_embeddings(parsed_hmeq_fin,  label="hmeq financial_risk")
raw_emb_hmeq_soc  = get_raw_embeddings(parsed_hmeq_soc,  label="hmeq social_capital")



stage_ohe_adult_fin = make_stage_ohe(parsed_adult_fin)
stability_adult_fin = make_stability_col(parsed_adult_fin)
stage_ohe_adult_soc = make_stage_ohe(parsed_adult_soc)
stability_adult_soc = make_stability_col(parsed_adult_soc)

stage_ohe_hmeq_fin  = make_stage_ohe(parsed_hmeq_fin)
stability_hmeq_fin  = make_stability_col(parsed_hmeq_fin)
stage_ohe_hmeq_soc  = make_stage_ohe(parsed_hmeq_soc)
stability_hmeq_soc  = make_stability_col(parsed_hmeq_soc)



print("ABLATION - ADULT FINANCIAL RISK FRAMING")

adult_fin_scores, _ = nested_cv_variant(
    X_raw          = X_raw_adult,
    raw_embeddings = raw_emb_adult_fin,
    stage_ohe      = stage_ohe_adult_fin,
    stability_col  = stability_adult_fin,
    y              = y_adult,
    n_pca          = best_n_adult,
    label          = "adult_financial_risk"
)


print("ABLATION - ADULT SOCIAL CAPITAL FRAMING")

adult_soc_scores, _ = nested_cv_variant(
    X_raw          = X_raw_adult,
    raw_embeddings = raw_emb_adult_soc,
    stage_ohe      = stage_ohe_adult_soc,
    stability_col  = stability_adult_soc,
    y              = y_adult,
    n_pca          = best_n_adult,
    label          = "adult_social_capital"
)


print("ABLATION - HMEQ FINANCIAL RISK FRAMING")

hmeq_fin_scores, _ = nested_cv_variant(
    X_raw          = X_raw_hmeq,
    raw_embeddings = raw_emb_hmeq_fin,
    stage_ohe      = stage_ohe_hmeq_fin,
    stability_col  = stability_hmeq_fin,
    y              = y_hmeq,
    n_pca          = best_n_hmeq,
    label          = "hmeq_financial_risk"
)

print("ABLATION - HMEQ SOCIAL CAPITAL FRAMING")

hmeq_soc_scores, _ = nested_cv_variant(
    X_raw          = X_raw_hmeq,
    raw_embeddings = raw_emb_hmeq_soc,
    stage_ohe      = stage_ohe_hmeq_soc,
    stability_col  = stability_hmeq_soc,
    y              = y_hmeq,
    n_pca          = best_n_hmeq,
    label          = "hmeq_social_capital"
)



print("PROMPT ABLATION SUMMARY")


print(f"\n{'dataset':<8} {'framing':<20} {'hybrid AUC':<14} {'vs baseline p':<16} {'significant'}")


ablation_rows = [
    ("adult", "career_trajectory", adult_hybrid_scores,  adult_baseline_scores),
    ("adult", "financial_risk",    adult_fin_scores,     adult_baseline_scores),
    ("adult", "social_capital",    adult_soc_scores,     adult_baseline_scores),
    ("hmeq",  "career_trajectory", hmeq_hybrid_scores,   hmeq_baseline_scores),
    ("hmeq",  "financial_risk",    hmeq_fin_scores,      hmeq_baseline_scores),
    ("hmeq",  "social_capital",    hmeq_soc_scores,      hmeq_baseline_scores),
]

for dataset, framing, hybrid_s, base_s in ablation_rows:
    a = np.array(base_s)
    b = np.array(hybrid_s)
    _, p = wilcoxon(a, b)
    sig = "yes" if p < 0.05 else "no"
    print(f"{dataset:<8} {framing:<20} {b.mean():<14.4f} {p:<16.4f} {sig}")

In [ ]:


fig, axes = plt.subplots(1, 2, figsize=(14, 6))


variants      = ["baseline", "llm only", "hybrid"]
adult_means   = [
    np.mean(adult_baseline_scores),
    np.mean(adult_llm_scores),
    np.mean(adult_hybrid_scores)
]
adult_stds    = [
    np.std(adult_baseline_scores),
    np.std(adult_llm_scores),
    np.std(adult_hybrid_scores)
]
hmeq_means    = [
    np.mean(hmeq_baseline_scores),
    np.mean(hmeq_llm_scores),
    np.mean(hmeq_hybrid_scores)
]
hmeq_stds     = [
    np.std(hmeq_baseline_scores),
    np.std(hmeq_llm_scores),
    np.std(hmeq_hybrid_scores)
]

colors = ["#94a3b8", "#f59e0b", "#2563eb"]
x      = np.arange(len(variants))
width  = 0.5


bars = axes[0].bar(x, adult_means, width, yerr=adult_stds,
                    color=colors, capsize=5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(variants)
axes[0].set_ylabel("AUC (nested cv mean)")
axes[0].set_title("UCI Adult Income - results")
axes[0].set_ylim(0.7, 1.0)

for bar, mean in zip(bars, adult_means):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                  bar.get_height() + 0.005,
                  f"{mean:.4f}", ha="center", va="bottom", fontsize=9)


_, p_adult = wilcoxon(adult_baseline_scores, adult_hybrid_scores)
sig_text_adult = f"hybrid vs baseline\np={p_adult:.4f} (not significant)" if p_adult >= 0.05 \
                  else f"hybrid vs baseline\np={p_adult:.4f} significant"
axes[0].text(0.98, 0.02, sig_text_adult,
              transform=axes[0].transAxes,
              ha="right", va="bottom", fontsize=8,
              bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))


bars2 = axes[1].bar(x, hmeq_means, width, yerr=hmeq_stds,
                     color=colors, capsize=5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(variants)
axes[1].set_ylabel("AUC (nested cv mean)")
axes[1].set_title("HMEQ Loan Default - results")
axes[1].set_ylim(0.3, 1.0)

for bar, mean in zip(bars2, hmeq_means):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                  bar.get_height() + 0.005,
                  f"{mean:.4f}", ha="center", va="bottom", fontsize=9)

_, p_hmeq = wilcoxon(hmeq_baseline_scores, hmeq_hybrid_scores)
sig_text_hmeq = f"hybrid vs baseline\np={p_hmeq:.4f} (not significant)" if p_hmeq >= 0.05 \
                 else f"hybrid vs baseline\np={p_hmeq:.4f} significant"
axes[1].text(0.98, 0.02, sig_text_hmeq,
              transform=axes[1].transAxes,
              ha="right", va="bottom", fontsize=8,
              bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.suptitle("LLM hybrid features vs baseline - nested CV results",
              fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{DRIVE_FOLDER}/final_results_chart.png", dpi=150)
plt.show()



fig, axes = plt.subplots(1, 2, figsize=(14, 5))

folds = list(range(1, 16))

axes[0].plot(folds, adult_baseline_scores, "o-", label="baseline",
              color="#94a3b8", linewidth=2)
axes[0].plot(folds, adult_hybrid_scores,   "o-", label="hybrid",
              color="#2563eb", linewidth=2)
axes[0].set_xlabel("fold")
axes[0].set_ylabel("AUC")
axes[0].set_title("adult - fold by fold AUC")
axes[0].legend()
axes[0].set_ylim(0.7, 1.0)

axes[1].plot(folds, hmeq_baseline_scores, "o-", label="baseline",
              color="#94a3b8", linewidth=2)
axes[1].plot(folds, hmeq_hybrid_scores,   "o-", label="hybrid",
              color="#2563eb", linewidth=2)
axes[1].set_xlabel("fold")
axes[1].set_ylabel("AUC")
axes[1].set_title("hmeq - fold by fold AUC")
axes[1].legend()
axes[1].set_ylim(0.3, 1.0)

plt.suptitle("fold by fold AUC comparison - baseline vs hybrid",
              fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{DRIVE_FOLDER}/fold_comparison_chart.png", dpi=150)
plt.show()



print("FINAL RESULTS SUMMARY")


print(f"\nUCI Adult Income (n=1000, nested cv 3x5=15 folds)")
print(f"  baseline AUC : {np.mean(adult_baseline_scores):.4f} +/- {np.std(adult_baseline_scores):.4f}")
print(f"  llm only AUC : {np.mean(adult_llm_scores):.4f} +/- {np.std(adult_llm_scores):.4f}")
print(f"  hybrid AUC   : {np.mean(adult_hybrid_scores):.4f} +/- {np.std(adult_hybrid_scores):.4f}")
print(f"  hybrid vs baseline p={p_adult:.4f} - {'not significant' if p_adult >= 0.05 else 'significant'}")

print(f"\nHMEQ Loan Default (n=1000, nested cv 3x5=15 folds)")
print(f"  baseline AUC : {np.mean(hmeq_baseline_scores):.4f} +/- {np.std(hmeq_baseline_scores):.4f}")
print(f"  llm only AUC : {np.mean(hmeq_llm_scores):.4f} +/- {np.std(hmeq_llm_scores):.4f}")
print(f"  hybrid AUC   : {np.mean(hmeq_hybrid_scores):.4f} +/- {np.std(hmeq_hybrid_scores):.4f}")
print(f"  hybrid vs baseline p={p_hmeq:.4f} - {'not significant' if p_hmeq >= 0.05 else 'significant'}")

print(f"\nall charts saved to {DRIVE_FOLDER}")
